In [0]:
# Databricks notebook source
# NVE HydAPI -> Bronze Delta ingestion

### Import libraries

In [0]:
import time
from datetime import date, timedelta
from itertools import islice
import json

import requests
from pyspark.sql import functions as F

### Set variables

In [0]:
HYDAPI_API_KEY = "NStth9G2fUOeXyl7WWr+Xw=="

In [0]:
# Destination for the Delta tables.
dbutils.widgets.text("catalog", "dariusz_dbw")
dbutils.widgets.text("schema", "bronze")

# Measurements: 1000 = water level; 1001 = discharge.
dbutils.widgets.text("parameters", "1000,1001")

# Leave empty to select stations automatically.
# Alternatively, enter comma-separated station IDs.
dbutils.widgets.text("station_ids", "")
dbutils.widgets.text("max_discovered_stations", "5")

# Resolution in minutes: 1440 = daily; 60 = hourly; 0 = instantaneous.
dbutils.widgets.text("resolution_time", "1440")

# Last seven days. Takes precedence over start_date and end_date.
dbutils.widgets.text("reference_time", "P7D/")

# Used only when reference_time is empty.
dbutils.widgets.text("start_date", "")
dbutils.widgets.text("end_date", "")

# Maximum calendar days per request when explicit dates are used.
dbutils.widgets.text("chunk_days", "31")

# Maximum stations and parameters included in each observation request.
dbutils.widgets.text("station_batch_size", "5")
dbutils.widgets.text("parameter_batch_size", "2")

BASE_URL = "https://hydapi.nve.no/api/v1"
SOURCE_SYSTEM = "NVE_HYDAPI"

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")

parameters_table = f"{catalog}.{schema}.nve_hydapi_parameters"
stations_table = f"{catalog}.{schema}.nve_hydapi_stations"
series_table = f"{catalog}.{schema}.nve_hydapi_series"
observations_table = f"{catalog}.{schema}.nve_hydapi_observations"

### Common functions

In [0]:
def get_api_key():
    """
    Read the HydAPI API key from a notebook variable.

    Inputs:
        No function arguments.
        Reads the global HYDAPI_API_KEY variable.

    Returns:
        str: The API key with surrounding whitespace removed.

    Behavior:
        Validates that an API key has been entered.
        No Databricks Secrets or Key Vault access is required.

    Raises:
        ValueError: If the key is empty or still contains the placeholder.
    """
    api_key = HYDAPI_API_KEY.strip()

    if not api_key or api_key == "PASTE_YOUR_HYDAPI_API_KEY_HERE":
        raise ValueError(
            "Set HYDAPI_API_KEY in the notebook before running ingestion."
        )

    return api_key


def csv_widget(name):
    """
    Read a comma-separated Databricks widget as a list.

    Inputs:
        name (str): Name of an existing Databricks widget.

    Returns:
        list[str]: Values with surrounding whitespace and empty entries
        removed. An empty widget produces an empty list.

    Behavior:
        Reads the widget, splits its contents at commas, and cleans
        each value. Values remain strings.

    Example:
        Widget contents: "1000, 1001"
        Returned value: ["1000", "1001"]
    """
    return [
        value.strip()
        for value in dbutils.widgets.get(name).split(",")
        if value.strip()
    ]


def chunked(values, size):
    """
    Split an iterable into batches.

    Inputs:
        values (iterable): Values to divide into batches.
        size (int): Maximum number of values in each batch.
            Must be greater than zero.

    Returns:
        A generator yielding lists containing at most size values.
        The final batch may contain fewer values.

    Behavior:
        Consumes the iterable progressively using islice().
        Used to limit the stations and parameters in each API request.

    Raises:
        ValueError: If size is not positive.

    Example:
        list(chunked(["a", "b", "c"], 2))
        returns [["a", "b"], ["c"]].
    """
    if size <= 0:
        raise ValueError("Batch size must be greater than zero.")

    iterator = iter(values)

    while batch := list(islice(iterator, size)):
        yield batch


def hydapi_get(endpoint, api_key, params=None, max_retries=5):
    """
    Send an authenticated GET request to HydAPI.

    Inputs:
        endpoint (str): Endpoint relative to BASE_URL, for example
            "Parameters", "Stations", "Series", or "Observations".
        api_key (str): HydAPI key retrieved from Databricks Secrets.
        params (dict or None): Optional HTTP query parameters.
        max_retries (int): Maximum total number of request attempts,
            including the first attempt. Defaults to 5.

    Returns:
        dict: The JSON response decoded into a Python dictionary.

    Behavior:
        Adds the X-API-Key and Accept headers.
        Uses a 60-second timeout.
        Retries HTTP 429 responses with a delay.
        Other HTTP errors and network errors are raised immediately.

        This function does not paginate or automatically split large
        requests. The calling code controls batches and date windows.

    Raises:
        ValueError: If max_retries is not positive or HTTP 400 occurs.
        requests.HTTPError: For other unsuccessful HTTP responses,
            including HTTP 429 after the last attempt.
        requests.RequestException: For connection or timeout failures.
        JSON decoding errors: If the response is not valid JSON.
    """
    if max_retries <= 0:
        raise ValueError("max_retries must be greater than zero.")

    url = f"{BASE_URL}/{endpoint.lstrip('/')}"
    headers = {
        "Accept": "application/json",
        "X-API-Key": api_key,
    }

    for attempt in range(max_retries):
        response = requests.get(
            url,
            headers=headers,
            params=params,
            timeout=60,
        )

        if response.status_code == 429 and attempt < max_retries - 1:
            reset_after = response.headers.get("x-rate-limit-reset")

            sleep_seconds = (
                int(reset_after)
                if reset_after and reset_after.isdigit()
                else 2 ** attempt
            )

            time.sleep(min(sleep_seconds, 60))
            continue

        if response.status_code == 400:
            raise ValueError(
                "HydAPI rejected the request. Check the parameters "
                "and try smaller station batches, parameter batches, "
                f"or date windows. Response: {response.text[:1000]}"
            )

        response.raise_for_status()
        return response.json()


def build_reference_times(
    reference_time,
    start_date,
    end_date,
    chunk_days,
):
    """
    Build the ReferenceTime values for observation requests.

    Inputs:
        reference_time (str): Explicit HydAPI time expression, such as
            "-P7D/". If populated, it takes precedence over the dates.
        start_date (str): Start date in YYYY-MM-DD format, or "".
        end_date (str): End date in YYYY-MM-DD format, or "".
        chunk_days (int): Maximum number of calendar days per window.
            Must be greater than zero when explicit dates are used.

    Returns:
        list[str or None]:
            [reference_time] when an explicit expression is supplied.
            A list of date ranges when both dates are supplied.
            [None] when no time filter is supplied.

    Behavior:
        Divides explicit dates into consecutive calendar windows.
        A None value tells the caller to omit ReferenceTime and use
        the API's default behavior.

    Raises:
        ValueError: If only one date is provided, a date is invalid,
            the end precedes the start, or chunk_days is not positive.
    """
    if reference_time:
        return [reference_time]

    if not start_date and not end_date:
        return [None]

    if not start_date or not end_date:
        raise ValueError(
            "Set both start_date and end_date, or use reference_time."
        )

    if chunk_days <= 0:
        raise ValueError("chunk_days must be greater than zero.")

    start = date.fromisoformat(start_date)
    end = date.fromisoformat(end_date)

    if end < start:
        raise ValueError("end_date must be on or after start_date.")

    windows = []
    current = start

    while current <= end:
        window_end = min(
            current + timedelta(days=chunk_days - 1),
            end,
        )

        windows.append(
            f"{current.isoformat()}/{window_end.isoformat()}"
        )
        current = window_end + timedelta(days=1)

    return windows

def write_bronze(rows, table_name):
    """
    Convert API records through JSON and overwrite a Bronze Delta table.

    Inputs:
        rows (list[dict]): Nonempty list of API records.
        table_name (str): Fully qualified destination table name.

    Returns:
        pyspark.sql.DataFrame: Records written to the table,
            including ingested_at.

    Behavior:
        Serializes the records as a JSON array.
        Infers its schema using Spark's JSON parser.
        Preserves nested objects and arrays, including seriesList.
        Adds ingested_at and replaces the destination Delta table.

    Raises:
        ValueError: If rows is empty.
        TypeError: If records cannot be serialized as JSON.
        Spark errors: If JSON parsing or table writing fails.

    Note:
        Holds the complete JSON payload in notebook memory.
        Intended for the current bounded ingestion, not large backfills.
    """
    if not rows:
        raise ValueError(
            f"No records returned for {table_name}; table not overwritten."
        )

    payload = json.dumps(rows, allow_nan=False)

    # Infer nested types with Spark's JSON parser.
    schema_ddl = (
        spark.range(1)
        .select(F.schema_of_json(F.lit(payload)).alias("schema"))
        .first()["schema"]
    )

    # An explicit string schema avoids Python dictionary inference.
    raw_df = spark.createDataFrame(
        [(payload,)],
        schema="payload STRING",
    )

    df = (
        raw_df
        .select(
            F.explode(
                F.from_json(
                    F.col("payload"),
                    schema_ddl,
                    options={"mode": "FAILFAST"},
                )
            ).alias("record")
        )
        .select("record.*")
        .withColumn("ingested_at", F.current_timestamp())
    )

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table_name)
    )

    return df


### Main code

In [0]:
# COMMAND ----------
# Read settings

api_key = get_api_key()

catalog = dbutils.widgets.get("catalog").strip()
schema = dbutils.widgets.get("schema").strip()

parameters = csv_widget("parameters")
station_ids = csv_widget("station_ids")

resolution_time = dbutils.widgets.get("resolution_time").strip()
reference_time = dbutils.widgets.get("reference_time").strip()
start_date = dbutils.widgets.get("start_date").strip()
end_date = dbutils.widgets.get("end_date").strip()

chunk_days = int(dbutils.widgets.get("chunk_days"))
station_batch_size = int(dbutils.widgets.get("station_batch_size"))
parameter_batch_size = int(dbutils.widgets.get("parameter_batch_size"))
max_discovered_stations = int(
    dbutils.widgets.get("max_discovered_stations")
)

if not parameters:
    raise ValueError("Set at least one parameter.")

if station_batch_size <= 0 or parameter_batch_size <= 0:
    raise ValueError("Batch sizes must be greater than zero.")

if not station_ids and max_discovered_stations <= 0:
    raise ValueError(
        "Set station_ids, or set max_discovered_stations "
        "to a positive number."
    )

reference_times = build_reference_times(
    reference_time,
    start_date,
    end_date,
    chunk_days,
)

tables = {
    "parameters": f"{catalog}.{schema}.nve_hydapi_parameters",
    "stations": f"{catalog}.{schema}.nve_hydapi_stations",
    "series": f"{catalog}.{schema}.nve_hydapi_series",
    "observations": f"{catalog}.{schema}.nve_hydapi_observations",
}

quoted_catalog = catalog.replace("`", "``")
quoted_schema = schema.replace("`", "``")

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS "
    f"`{quoted_catalog}`.`{quoted_schema}`"
)

In [0]:
# COMMAND ----------
# Fetch parameter and station metadata

parameter_payload = hydapi_get("Parameters", api_key)
parameter_rows = parameter_payload.get("data", []) or []

station_payload = hydapi_get("Stations", api_key)
station_rows = station_payload.get("data", []) or []


# COMMAND ----------
# Fetch series metadata for the requested parameters

series_rows = []

for parameter in parameters:
    series_payload = hydapi_get(
        "Series",
        api_key,
        params={"Parameter": parameter},
    )

    series_rows.extend(series_payload.get("data", []) or [])

In [0]:
# COMMAND ----------
# Select stations automatically when no explicit IDs were supplied

if not station_ids:
    seen_station_ids = set()

    for series in series_rows:
        matching_resolution = any(
            str(resolution.get("resTime")) == resolution_time
            for resolution in series.get("resolutionList", []) or []
        )

        station_id = series.get("stationId")

        if (
            matching_resolution
            and station_id
            and station_id not in seen_station_ids
        ):
            station_ids.append(station_id)
            seen_station_ids.add(station_id)

        if len(station_ids) >= max_discovered_stations:
            break

    if not station_ids:
        raise ValueError(
            "No stations found for the requested parameters "
            "and resolution_time."
        )

In [0]:
# COMMAND ----------
# Fetch observations in station, parameter, and date batches

observation_rows = []

for station_batch in chunked(station_ids, station_batch_size):
    for parameter_batch in chunked(parameters, parameter_batch_size):
        for requested_reference_time in reference_times:
            request_params = {
                "StationId": ",".join(station_batch),
                "Parameter": ",".join(parameter_batch),
                "ResolutionTime": resolution_time,
            }

            if requested_reference_time:
                request_params["ReferenceTime"] = requested_reference_time

            payload = hydapi_get(
                "Observations",
                api_key,
                params=request_params,
            )

            for series in payload.get("data", []) or []:
                for observation in series.get("observations", []) or []:
                    observation_rows.append(
                        {
                            "stationId": series.get("stationId"),
                            "stationName": series.get("stationName"),
                            "parameter": series.get("parameter"),
                            "parameterName": series.get("parameterName"),
                            "parameterNameEng": series.get(
                                "parameterNameEng"
                            ),
                            "serieVersionNo": series.get("serieVersionNo"),
                            "method": series.get("method"),
                            "unit": series.get("unit"),
                            "observation_time": observation.get("time"),
                            "value": observation.get("value"),
                            "correction": observation.get("correction"),
                            "quality": observation.get("quality"),
                            "resolution_time": resolution_time,
                            "requested_reference_time": (
                                requested_reference_time
                            ),
                            "hydapi_created_at": payload.get("createdAt"),
                        }
                    )

In [0]:
# Write the Bronze Delta tables
df_parameters = write_bronze(
    parameter_rows,
    tables["parameters"],
)

df_stations = write_bronze(
    station_rows,
    tables["stations"],
)

df_series = write_bronze(
    series_rows,
    tables["series"],
)

df_observations = write_bronze(
    observation_rows,
    tables["observations"],
)
# COMMAND ----------
# Preview the ingested data

#display(df_parameters.limit(20))
#display(df_stations.limit(20))
#display(df_series.limit(20))
#display(df_observations.limit(20))